In [0]:
# Notebook parameters

params = {
    "proj_dir": ""
}

# create text widgets
for k in params.keys():
    dbutils.widgets.text(k, "", "")

# fetch values
for k in params.keys():
    params[k] = dbutils.widgets.get(k)
    print(k, ":", params[k])

In [0]:
import os
import time
from datetime import datetime
from tqdm.notebook import tqdm
import pydicom
import asyncio
from functools import lru_cache
from pyspark.sql import types as T
from pyspark.sql import functions as F
from glob import glob

In [0]:
ROOT_DIR = "/Volumes/1_inland/sectra/vone"
full_proj_path = os.path.join(ROOT_DIR, params["proj_dir"])

In [0]:
df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.dcm")
    .option("recursiveFileLookup", "true")
    .load(full_proj_path)
)
print(df.count())
display(df.limit(0))

In [0]:
from io import BytesIO
from pydicom.filebase import DicomFileLike

In [0]:
import pandas as pd
from pathlib import Path

def process_dicom_batch(pdf_iter):
    for pdf in pdf_iter:
        output_rows = []
        for path, content in zip(pdf['path'], pdf['content']):
            try:
                ds = pydicom.dcmread(BytesIO(content), force=True)
                #ds.decompress()

                try:
                    accession_nbr = str(ds["AccessionNumber"].value)
                except:
                    accession_nbr = None

                try:
                    study_id = str(ds["StudyID"].value)
                except:
                    study_id = None

                try:
                    if "ProcedureCodeSequence" in ds:
                        code_value = ds.ProcedureCodeSequence[0].CodeValue
                    elif "RequestedProcedureCodeSequence" in ds:
                        seq = ds.RequestedProcedureCodeSequence
                        if len(seq) > 0 and "CodeValue" in seq[0]:
                            code_value = seq[0].CodeValue
                        else:
                            code_value = None
                    else:
                        code_value = ds.RequestedProcedureID
                except:
                    code_value = None

                try:
                    description = str(ds["StudyDescription"].value)
                except:
                    description = None
                
                output_rows.append((path, accession_nbr, study_id, code_value, description))
            except:
                output_rows.append((path, None, None, None, None))


        yield pd.DataFrame(output_rows, columns=["path", "accession_number", "study_id", "code_value", "description"])

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BinaryType

# Define output schema: path + redacted DICOM binary
output_schema = StructType([
    StructField("path", StringType(), True),
    StructField("accession_number", StringType(), True),
    StructField("study_id", StringType(), True),
    StructField("code_value", StringType(), True),
    StructField("description", StringType(), True)
])

md_df = df.mapInPandas(process_dicom_batch, schema=output_schema)
display(md_df.limit(10))

